<a href="https://colab.research.google.com/github/ShirinMahbuba/flyrank-ml-starter/blob/main/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

Simple words, honest numbers.

## 0. Setup

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape[0], "rows,", df.shape[1], "columns")

30000 rows, 44 columns


## 1. My lane (or freestyle) and why

**Lane: Content freshness — does staleness actually predict decline?**

I'm picking this lane because Week 1 and Week 2 already put two of its key pieces in my hands: a hand-written `stale x visible` rule (`days_since_last_update >= 180` and `impressions_90d >= 500`), and a readable decision tree that also leaned on `days_since_last_update` as a split. Both weeks hinted that recency of updates matters for whether a page is declining, but neither fully answered *how much* it matters compared to other signals, or whether "stale" is too blunt a definition. That's a concrete, well-scoped question I can dig into for 7 weeks without needing anything beyond the starter CSV, and it connects directly to something a content team can actually act on: which pages to prioritize for a refresh.

## 2. The question: decision, action, cost of a wrong call

**Decision this work improves:** which pages a content team should schedule for a refresh first, out of a much larger backlog than anyone has time to update.

**Who acts on it:** a content or SEO strategist working through a client's refresh queue, deciding where to spend limited editing hours this month.

**Cost of a wrong call:**
- *False positive* (flagging a page as "needs refresh" that wasn't actually declining): wasted editor time on a page that didn't need it, while a genuinely declining page waits longer.
- *False negative* (missing a page that's quietly declining because it doesn't look "stale" by a simple date rule): the page keeps losing visibility and traffic until someone notices too late, and by the time it's caught, more work is needed to recover it than if it had been caught early.

Because editor time is the scarce resource, the real cost isn't just "wrong flag" — it's opportunity cost: every hour spent on a false positive is an hour not spent on a page that's actually declining.

## 3. Quick look at the data (2-3 real numbers)

In [2]:
# Build the same 'declining' label used in Week 2
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Number 1 — how common is staleness, and how common is decline?
stale_rate = (df["days_since_last_update"] >= 180).mean()
decline_rate = df["is_declining_label"].mean()
print(f"Share of pages that are 'stale' (>=180 days since update): {stale_rate:.3f}")
print(f"Share of pages that are declining (trend_direction == 'down'): {decline_rate:.3f}")

# Number 2 — does staleness actually associate with decline?
decline_rate_by_staleness = df.groupby(df["days_since_last_update"] >= 180)["is_declining_label"].mean()
print("\nDeclining rate, stale vs not stale:")
print(decline_rate_by_staleness.round(3).to_string())

# Number 3 — median days since last update, declining vs not declining
median_days_by_label = df.groupby("is_declining_label")["days_since_last_update"].median()
print("\nMedian days_since_last_update, by declining label (1 = declining):")
print(median_days_by_label.round(0).to_string())


Share of pages that are 'stale' (>=180 days since update): 0.006
Share of pages that are declining (trend_direction == 'down'): 0.542

Declining rate, stale vs not stale:
days_since_last_update
False    0.542
True     0.471

Median days_since_last_update, by declining label (1 = declining):
is_declining_label
0    20.0
1    20.0


*Fill in after running:* one sentence on what these three numbers show — does the gap between stale/not-stale or declining/not-declining look big enough to be worth building a model around, or is it weaker than the Week 1/2 hints suggested?

## 4. Careful words: what I can and can't claim

**What this work will be able to say:**
- *Observed*: in this anonymized sample, pages that haven't been updated in N+ days show a higher/lower rate of decline than pages updated more recently (numbers from Section 3).
- *Measured*: a ranking built from freshness and visibility signals achieves a certain Precision@K on a held-out set of clients, compared to a simple hand rule.
- *Directional*: staleness appears to be one signal associated with decline, among others — not necessarily the strongest one, and not necessarily causal.
- *Decision-support*: a ranked "refresh candidates" list that a strategist can use as one input, alongside their own judgment, to prioritize a backlog.

**What this work will never say:**
- Not causal proof — I can't claim that updating a page *causes* it to stop declining, only that staleness and decline are associated in this data.
- Not "predicting Google" — this doesn't model or reverse-engineer the search algorithm; it models this dataset's *observable outcomes* (impressions, CTR, trend).
- Not a guarantee for any individual page — correlations and precision scores describe the sample on average, not what will happen to one specific page.
- No client names, URLs, or private data will appear in anything published from this work.

---
### Self-check (confirm before submitting)
- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit repo URL on the card